# Evaluation explorer

Loads `reports/evaluation/en/evaluation_results.json` (written by `python evaluate.py`) and reproduces the main tables and plots. Nothing here changes the model.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
R = json.loads((ROOT / 'reports/evaluation/en/evaluation_results.json').read_text())
R['model_version']

## Held-out test metrics

In [ ]:
t = R['test']['overall']
pd.DataFrame({k: [t[k], (t['ci95'] or {}).get(k)] for k in ['accuracy','precision','recall','f1','roc_auc','pr_auc','fpr','fnr']}, index=['value','95% CI']).T

## Cross-generator and cross-domain generalisation

In [ ]:
rows = []
for kind in ('cross_generator', 'cross_domain'):
    for k, r in R.get(kind, {}).items():
        rows.append({'study': kind, 'held_out': k, 'auc_held_out': r['held_out']['roc_auc'], 'auc_in_distribution': r['in_distribution_reference']['roc_auc'], 'fpr_held_out': r['held_out']['fpr']})
pd.DataFrame(rows)

## Reliability diagram (document level)

In [ ]:
from aidetect.visualization import SERIES
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], ls=':', c='#898781')
for i, m in enumerate(['none', 'platt', 'isotonic']):
    bins = [b for b in R['calibration']['document']['test'][m]['reliability']['bins'] if b['count']]
    ax.plot([b['mean_pred'] for b in bins], [b['frac_pos'] for b in bins], marker='o', c=SERIES[i], label=m)
ax.set_xlabel('mean predicted'); ax.set_ylabel('observed AI share'); ax.legend(frameon=False)

## False-positive audit

In [ ]:
pd.DataFrame(R['fp_audit']['groups']).T[['n', 'writer_group', 'genre', 'fpr', 'mean_probability']]

## Adversarial categories

In [ ]:
pd.DataFrame(R['adversarial']).T[['n_ai', 'n_human', 'ai_detection_rate', 'human_flag_rate', 'roc_auc']]